# 03 — Experimental grass-shaping ablation
**WARNING:** The grayscale darkness heuristic is unvalidated and may mistake road for grass. Do not run this until you have manually validated representative frames. No forced termination; original evaluation returns remain unchanged. Use separate output folders.

In [ ]:
from pathlib import Path
import subprocess, sys, zipfile, os
from google.colab import drive, files
drive.mount('/content/drive')
REPO_URL='https://github.com/MilkySon/RL_carracing_competition.git'
BRANCH='261030'
REPO=Path('/content/RL_carracing_competition')
BASE=REPO/'experiments'/'continuous_ablation'/'baseline'
V2=Path('/content/CarRacing_Continuous_v2_1')
ROOT=Path('/content/drive/MyDrive/CarRacing_Continuous_v2_1')
if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,REPO_URL,str(REPO)],check=True)
assert (BASE/'train.py').exists(), f'Baseline missing: {BASE}'
if not V2.exists():
    print('Upload CarRacing_Continuous_v2_1.zip from your computer')
    uploaded=files.upload()
    name=next((x for x in uploaded if x.endswith('.zip') and 'Continuous_v2_1' in x),None)
    assert name, 'Please upload CarRacing_Continuous_v2_1.zip'
    with zipfile.ZipFile(name) as z:z.extractall('/content')
assert (V2/'scripts'/'train_v2.py').exists()
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(BASE/'requirements.txt')],check=True)
subprocess.run([sys.executable,str(V2/'scripts'/'install_overlay.py'),'--baseline',str(BASE)],check=True)
print('Baseline:',BASE,'Overlay:',V2,'Output:',ROOT)

In [ ]:
subprocess.run([sys.executable,str(V2/'scripts'/'test_boundaries.py')],check=True)
sys.path.insert(0,str(V2/'scripts'))
from test_boundaries import verify_implementation
verify_implementation(BASE)
probe_path=ROOT/'diagnostics'/'forward_probe.csv'
subprocess.run([sys.executable,str(V2/'scripts'/'forward_probe.py'),
                '--baseline',str(BASE),'--out',str(probe_path),'--seed','1000','--steps','100'],check=True)
import pandas as pd
probe=pd.read_csv(probe_path)
display(probe)
forward=float(probe.loc[probe.scenario=='forward','mean_speed'].iloc[0])
coast=float(probe.loc[probe.scenario=='coast','mean_speed'].iloc[0])
assert forward > coast + 0.1, 'FORWARD GATE FAILED: investigate environment and action mapping before training'
print('PASS: forward action increases speed compared with coasting')

In [ ]:
# Explicit opt-in: first inspect and validate the grass detector on representative road/grass observations.
GRASS_DETECTOR_VALIDATED=False
assert GRASS_DETECTOR_VALIDATED, 'Manually validate the detector before enabling shaping.'

In [ ]:
AGENT='e2_v2' # choose e1_v2, e2_v2, e3_v2, sac_v2, td3_v2
SEED=0
BUDGET=300000
EVAL_EVERY=50000
SHAPING=True
RUN=ROOT/AGENT/f'seed_{SEED}_'+('shaped' if SHAPING else 'raw')
# Fail rather than silently overwrite prior experiment
assert not any((RUN/x).exists() for x in ['train.csv','eval.csv','latest.pt']), f'Existing run: {RUN}'
cmd=[sys.executable,str(V2/'scripts'/'train_v2.py'),'--baseline',str(BASE),
     '--agent',AGENT,'--seed',str(SEED),'--budget',str(BUDGET),
     '--eval-every',str(EVAL_EVERY),'--run-dir',str(RUN)]
if SHAPING:cmd.append('--shaping')
print('Running:', ' '.join(cmd))
subprocess.run(cmd,check=True)